# Collate sequence

This script collates all output files produced by the pipeline for a given sequence of plates, into a single table.

This table can be displayed by notebook *pipeline_results.ipynb*.

In [1]:
import os
from importlib import reload

import numpy as np

from astropy.table import Table, join, hstack, vstack

import settings
from settings import get_parameters, fname, sequences, current_sequence
from library import update_dataset

In [2]:
# input sequence
seq_key = current_sequence
sequence = sequences[seq_key]

In [3]:
# build collated table
table_collated = None

print("Sequence: ", sequence)

for i in range(len((sequence)) - 1):
    
    plate_id_str = str(sequence[i])
    next_plate_id_str = str(sequence[i+1])

    key = plate_id_str + ',' + next_plate_id_str
    print("Processing dataset: ", key)
    
    update_dataset(key)
    reload(settings)
    from settings import current_dataset
    
    par = get_parameters(current_dataset)
    
    table_candidates = Table.read(fname(par['table_candidates']), format='fits')
    print("Rows in dataset: ", key, "  ", len(table_candidates))
    
    if table_collated is None:
        table_collated = table_candidates
    else:    
        table_collated = vstack([table_collated, table_candidates])

Sequence:  [9529, 9553, 9555]
Processing dataset:  9529,9553
Rows in dataset:  9529,9553    1
Processing dataset:  9553,9555
Rows in dataset:  9553,9555    61


TableMergeError: The 'natmag' columns have incompatible types: ['float64', 'bytes80']

In [7]:
table_candidates['natmag']

9.909739
10.058538
10.182825
10.176364
10.251848
10.7382145
11.056556
10.742153
11.304646
11.186846
11.259071


Code below is no longer needed, but we keep it in place in case we need to tweak the output table, by removing additional sources from it.

In [ ]:
source_id_list = [(sid,plate_id) for (sid,plate_id) in zip(table_collated['source_id'], table_collated['plate_id_1'])]

In [ ]:
source_id_list

In [ ]:
mask = np.isin(table_collated['source_id'], source_id_list)

table_final = table_collated[mask]

out_file = fname('pipeline_final_' + seq_key + '.fits')
table_final.write(out_file, format='fits', overwrite=True)

In [ ]:
print("Created output file: ", out_file)

In [ ]:
table_final